# 최종 생성 4b0327a + Golden Set v3.1 + 채점기 v3.1.2 + 기존 Qwen3-VL 근거 재사용

생성 브랜치의 2026-09-10 최종 커밋 `4b0327a`를 고정해 사용합니다. `4a19d50` 기준에 표기 정규화, 규칙 기반 누락 보완, 표 답변 후처리가 추가된 버전입니다. VLM 서버를 새로 실행하거나 HWP/PDF 이미지를 다시 추출하지 않고, 이전 실험에서 정상 생성된 시각 근거 4건을 재사용합니다.

채점은 v3.1.0의 점수 기준을 유지하면서, v3.1.2로 파일명 내부 대괄호까지 보존하며 인용 상태를 `valid / misplaced / missing`으로 분리하고 실패 원인 후보를 기록합니다.

> 주의: 관련 이미지가 이미 선택된 상태에서 VLM 근거를 연결한 효과를 검증합니다. 정답 위치를 모르는 자동 시각 검색 성능을 뜻하지 않습니다.

커널은 `myenv`를 선택하고 `Kernel > Restart Kernel` 후 위에서부터 실행합니다.

In [ ]:
# 0. 팀원 생성 브랜치의 최신 검증 커밋을 원본 수정 없이 메모리에 로드
import subprocess, sys, types
from pathlib import Path

ROOT = Path('/home/kongseok/sprint-public-procurement-rag-assistant')
GENERATION_SOURCE_COMMIT = '4b0327a517ee2e5e7fb8c35c4f81c0b206ff5428'
GENERATION_SOURCE_BRANCH = 'origin/feat/generation-pipeline-prompt-eval'

def git_object_exists(commit):
    return subprocess.run(['git', 'cat-file', '-e', commit + '^{commit}'], cwd=ROOT, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0

if not git_object_exists(GENERATION_SOURCE_COMMIT):
    subprocess.run(['git', 'fetch', 'origin', 'feat/generation-pipeline-prompt-eval'], cwd=ROOT, check=True)
assert git_object_exists(GENERATION_SOURCE_COMMIT), '고정한 최신 생성 커밋을 가져오지 못했습니다.'

def source_at(path):
    return subprocess.check_output(['git', 'show', f'{GENERATION_SOURCE_COMMIT}:{path}'], cwd=ROOT, text=True, encoding='utf-8')

prompt_name = 'src.generation.generation_prompts'
prompt_module = types.ModuleType(prompt_name)
prompt_module.__file__ = f'git:{GENERATION_SOURCE_COMMIT}:src/generation/generation_prompts.py'
exec(compile(source_at('src/generation/generation_prompts.py'), prompt_module.__file__, 'exec'), prompt_module.__dict__)
sys.modules[prompt_name] = prompt_module

# 기존 스냅샷 import 경로에 최신 코드를 주입해 아래 실험 셀을 그대로 사용합니다.
answer_name = 'experiments.dahye_latest_20260909.answer_generation'
answer_module = types.ModuleType(answer_name)
answer_module.__file__ = f'git:{GENERATION_SOURCE_COMMIT}:src/generation/answer_generation.py'
exec(compile(source_at('src/generation/answer_generation.py'), answer_module.__file__, 'exec'), answer_module.__dict__)
sys.modules[answer_name] = answer_module
print('최종 생성 코드 로드:', GENERATION_SOURCE_COMMIT)
print('반영 내용: 100분의 N·N% 표기 정규화 + 누락 키워드 보완 + 신인도 가점표 후처리')

In [ ]:
# 1. 실행 환경 및 필수 파일 확인
import getpass, hashlib, json, os, re, time
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace

ROOT = Path('/home/kongseok/sprint-public-procurement-rag-assistant')
assert ROOT.exists(), f'레포 경로가 없습니다: {ROOT}'
os.chdir(ROOT)
os.environ['CUDA_VISIBLE_DEVICES'] = ''  # KURE는 CPU, 이 노트북은 vLLM/GPU 미사용

saved_key = os.environ.get('OPENAI_API_KEY', '').strip()
if not saved_key.isascii() or not saved_key.startswith('sk-'):
    saved_key = getpass.getpass('OpenAI API key: ').strip()
    os.environ['OPENAI_API_KEY'] = saved_key
assert saved_key.isascii() and saved_key.startswith('sk-'), '실제 OpenAI API 키를 입력하세요.'

EVIDENCE_PATH = ROOT / 'output/experiments/b_plan_v3_vlm/visual_evidence.jsonl'
required = [
    ROOT / 'output/chunks.pkl', ROOT / 'output/chroma_db', ROOT / 'output/merged_docs.pkl',
    ROOT / 'data/golden_set_v3/rag-56.draft.jsonl',
    ROOT / 'data/golden_set_v3/set-13.draft.jsonl',
    ROOT / 'data/golden_set_v3/document-structure-visual-qa.jsonl',
    ROOT / 'src/evaluation/scoring_v3/scorer_v3_1_2.py',
    ROOT / 'src/evaluation/golden_set_v3_1.py',
    ROOT / 'experiments/dahye_latest_20260909/answer_generation.py',
    ROOT / 'experiments/dahye_latest_20260909/generation_prompts.py', EVIDENCE_PATH,
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, '없는 파일:\n- ' + '\n- '.join(missing)
print('작업 위치:', ROOT)
print('VLM 서버/GPU: 사용하지 않음')
print('재사용할 시각 근거:', EVIDENCE_PATH)

In [ ]:
# 2. 기존 VLM 근거, Golden Set v3.1, 채점기와 검색기 로드
import pandas as pd
from openai import OpenAI
from experiments.dahye_latest_20260909.answer_generation import ask_rfp_v9
from src.data_processing.chunking import load_chunks
from src.evaluation.document_ids import extract_context_doc_ids
from src.evaluation.golden_set_v3_1 import GOLDEN_PATCH_VERSION, load_golden_set_v3_1
from src.evaluation.scoring_v3 import scorer_v3_1_2 as scorer
from src.retrieval.embeddings import SentenceTransformerEmbedding
from src.retrieval.indexing import HybridIndex

assert scorer.SCORER_VERSION == '3.1.2', scorer.SCORER_VERSION
with EVIDENCE_PATH.open(encoding='utf-8') as handle:
    evidence_rows = [json.loads(line) for line in handle if line.strip()]
evidence_by_case = {str(row['case_id']): row for row in evidence_rows}
assert len(evidence_by_case) == 4, f'기존 VLM 근거가 4건이 아닙니다: {len(evidence_by_case)}'
assert all(not row.get('error') and row.get('evidence_text') for row in evidence_rows), '기존 VLM 근거에 오류 또는 빈 값이 있습니다.'
assert all(row.get('model') == 'Qwen/Qwen3-VL-8B-Instruct' for row in evidence_rows), '예상한 VLM 모델의 근거가 아닙니다.'

chunks = load_chunks()
assert chunks, 'output/chunks.pkl을 읽지 못했습니다.'
child_chunks = [chunk for chunk in chunks if getattr(chunk, 'strategy', '') != 'parent']
corpus_doc_ids = {str(chunk.doc_id) for chunk in chunks}
golden = load_golden_set_v3_1(corpus_doc_ids=corpus_doc_ids)
golden_rows = golden.to_dict('records')
assert len(golden_rows) == 79, len(golden_rows)

doc_to_biz = {}
for chunk in chunks:
    metadata = getattr(chunk, 'metadata', {}) or {}
    business_name = metadata.get('사업명') or metadata.get('사업_명') or ''
    doc_to_biz.setdefault(str(chunk.doc_id), str(business_name))
catalog = sorted(doc_to_biz.items())
embedding_backend = SentenceTransformerEmbedding()
assert embedding_backend.name == 'nlpai-lab/KURE-v1', embedding_backend.name
index = HybridIndex(chunks, persist=True, embedding_backend=embedding_backend)
base_client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
print(f'VLM 근거 {len(evidence_by_case)}건, 문서 {len(corpus_doc_ids)}개, 청크 {len(chunks):,}개, 골든셋 {len(golden_rows)}문항')
print('채점기:', scorer.SCORER_VERSION, '/ 골든셋 패치:', GOLDEN_PATCH_VERSION)

In [ ]:
# 3. 기존 VLM 근거 주입 및 체크포인트 준비
class RecordingCompletions:
    def __init__(self, delegate, evidence=None):
        self.delegate, self.evidence, self.last_prompt = delegate, evidence or {}, ''
    def create(self, **kwargs):
        messages = [dict(message) for message in (kwargs.get('messages') or [])]
        evidence_text = str(self.evidence.get('evidence_text') or '')
        if messages and evidence_text:
            prompt = str(messages[-1].get('content', ''))
            visual_block = (
                f"[문서: {self.evidence.get('doc_id', '')}]\n"
                f"[기존 Qwen3-VL 시각 근거: {self.evidence.get('evidence_id', 'saved-visual-evidence')}]\n"
                f"{evidence_text}\n\n"
            )
            prompt = prompt.replace('## 질문', visual_block + '## 질문', 1)
            messages[-1]['content'] = prompt
            kwargs['messages'] = messages
        self.last_prompt = str(messages[-1].get('content', '')) if messages else ''
        return self.delegate.create(**kwargs)

class RecordingOpenAI:
    def __init__(self, client, evidence=None):
        self.completions = RecordingCompletions(client.chat.completions, evidence)
        self.chat = SimpleNamespace(completions=self.completions)

def extract_context(prompt):
    marker = '## 컨텍스트 (검색된 문서 조각)'
    if marker not in prompt: return ''
    return prompt.split(marker, 1)[1].split('## 질문', 1)[0].strip()

def cited_docs(answer):
    match = re.search(r'\[\s*근거\s*:\s*(.+?)\]\s*$', str(answer or ''), re.DOTALL)
    return [value.strip() for value in match.group(1).split(',') if value.strip()] if match else []

def retrieval_recall(expected, retrieved):
    expected_set = set(expected or [])
    return len(expected_set & set(retrieved)) / len(expected_set) if expected_set else None

def fact_coverage(row, context):
    groups = scorer._BASE.fact_groups(row)
    if not groups or not context: return None
    matched = sum(any(scorer.option_matches(context, option) for option in group) for group in groups)
    return matched / len(groups)

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = ROOT / 'output/generation_4b0327a_vlm_cached_v3_1_2_runs' / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = OUT_DIR / 'inference.jsonl'
manifest = {
    'run_id': RUN_ID, 'mode': 'saved_vlm_evidence_reuse',
    'generation_model': 'gpt-5-mini', 'visual_model': 'Qwen/Qwen3-VL-8B-Instruct',
    'visual_evidence_case_count': len(evidence_by_case),
    'visual_evidence_path': str(EVIDENCE_PATH),
    'visual_evidence_sha256': hashlib.sha256(EVIDENCE_PATH.read_bytes()).hexdigest(),
    'generation_source_branch': GENERATION_SOURCE_BRANCH,
    'generation_source_commit': GENERATION_SOURCE_COMMIT,
    'generation_parent_commit': '4a19d505bdfaa9cc2bfb24eec53bec6a4c107db9',
    'scorer_version': scorer.SCORER_VERSION,
    'golden_patch_version': GOLDEN_PATCH_VERSION,
}
(OUT_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('결과 폴더:', OUT_DIR)

In [ ]:
# 4. 79문항 생성: 성공 문항은 즉시 저장되고, 같은 셀 재실행 시 건너뜁니다.
def read_checkpoint():
    if not CHECKPOINT.exists(): return []
    with CHECKPOINT.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

prior = read_checkpoint()
completed = {str(row['id']) for row in prior if row.get('id') and not row.get('execution_error')}
print(f'기존 성공 {len(completed)}건, 남은 {len(golden_rows) - len(completed)}건')
for number, row in enumerate(golden_rows, start=1):
    case_id = str(row['id'])
    if case_id in completed: continue
    started = time.perf_counter()
    recording_client = RecordingOpenAI(base_client, evidence_by_case.get(case_id))
    answer, error = None, None
    try:
        answer = ask_rfp_v9(str(row['query']), recording_client, index, child_chunks, catalog, model_name='gpt-5-mini')
        if answer == '(답변 생성 실패)': error = 'answer_generation_returned_failure'
    except Exception as exc:
        error = f'{type(exc).__name__}: {exc}'
    context = extract_context(recording_client.completions.last_prompt)
    retrieved = extract_context_doc_ids(context)
    if not retrieved and answer: retrieved = cited_docs(answer)
    result = {
        'id': case_id, 'answer': answer, 'execution_error': error,
        'retrieved_doc_ids': retrieved,
        'retrieval_recall': retrieval_recall(row.get('expected_doc_id'), retrieved),
        'context_fact_coverage': fact_coverage(row, context),
        'elapsed_seconds': round(time.perf_counter() - started, 3),
        'visual_evidence_used': case_id in evidence_by_case,
        'visual_evidence_id': (evidence_by_case.get(case_id) or {}).get('evidence_id'),
        'visual_model': (evidence_by_case.get(case_id) or {}).get('model'),
    }
    with CHECKPOINT.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(result, ensure_ascii=False) + '\n')
    if error: raise RuntimeError(f'{case_id} 생성 실패: {error}. 원인을 고친 뒤 이 셀만 다시 실행하세요.')
    completed.add(case_id)
    print(f'[{number:02d}/{len(golden_rows)}] {case_id} 완료')
print('79문항 생성 완료')

In [ ]:
# 5. Golden Set v3.1 보정본과 채점기 v3.1.2로 최종 채점
latest = {}
for row in read_checkpoint():
    case_id = str(row.get('id', ''))
    if case_id and (case_id not in latest or not row.get('execution_error')):
        latest[case_id] = row

predictions = [latest[str(row['id'])] for row in golden_rows if str(row['id']) in latest]
assert len(predictions) == 79, f'완료 문항이 79개가 아닙니다: {len(predictions)}'
assert not any(row.get('execution_error') for row in predictions), '실행 오류 문항이 남아 있습니다.'

details, summary = scorer.evaluate(golden_rows, predictions)
prediction_by_id = {str(row['id']): row for row in predictions}
golden_by_id = {str(row['id']): row for row in golden_rows}

for detail in details:
    case_id = str(detail['id'])
    prediction = prediction_by_id[case_id]
    detail['source_lane'] = golden_by_id[case_id].get('source_lane')
    detail['visual_evidence_used'] = bool(prediction.get('visual_evidence_used'))
    detail['visual_evidence_id'] = prediction.get('visual_evidence_id')
    detail['visual_model'] = prediction.get('visual_model')

def average(rows, field):
    values = [float(row[field]) for row in rows if row.get(field) is not None]
    return sum(values) / len(values) if values else None

visual_details = [row for row in details if row.get('source_lane') == 'visual']
vlm_details = [row for row in visual_details if row.get('visual_evidence_used')]
summary.update({
    **manifest,
    'output_directory': str(OUT_DIR),
    'gold_document_locator_used': 'precomputed evidence; not a blind visual retrieval run',
    'visual_evaluation': {
        'total_visual_cases': len(visual_details),
        'saved_vlm_evidence_cases': len(vlm_details),
        'visual_end_to_end_score': average(visual_details, 'end_to_end_score'),
        'saved_vlm_case_end_to_end_score': average(vlm_details, 'end_to_end_score'),
    },
})

scorer.write_jsonl(OUT_DIR / 'scored_details.jsonl', details)
pd.DataFrame(details).to_csv(
    OUT_DIR / 'scored_details.csv', index=False, encoding='utf-8-sig'
)
(OUT_DIR / 'summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
display(pd.DataFrame([summary]).T.rename(columns={0: '결과'}))
print('결과 폴더:', OUT_DIR)
print('주의: 기존 VLM 근거 4건 재사용 결과이며 blind visual retrieval 평가는 아닙니다.')
print('생성 소스:', GENERATION_SOURCE_COMMIT, '/ 채점기:', scorer.SCORER_VERSION)